<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V1 Dynamic Re-centering Grid + Risk Guardrails

V1 now **inherits the frozen V0-B starting portfolio and fixed order size**. This makes V0 vs V1 comparable at time zero.

V1 changes only what happens after initialization:
- Active grid: 30 grids, 1,000 USDT gap, initial range 27,000–57,000, reference 42,000.
- Re-center when candle close moves ±5 grids from the current reference.
- Old positions survive and keep their original sell targets.
- Risk guardrails apply to **new V1 BUYs only**. The inherited V0-B starting inventory is not forced to satisfy those limits at time zero.
- No compounding. Live execution remains OFF.


# 0. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64, bisect, heapq, json, os, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests


# 1. Trading System

In [ ]:
SYMBOL = "BTCUSDT"
INITIAL_CAPITAL = 3000.0

# Frozen V0-B initialization
V0_BASELINE_FLOOR = 38000.0
V0_BASELINE_CEILING = 127000.0

# V1 active grid
V1_INITIAL_FLOOR = 27000.0
V1_INITIAL_CEILING = 57000.0
GRID_GAP = 1000.0
BUY_FEE = 0.001
SELL_FEE = 0.001
RECENTER_TRIGGER_GRIDS = 5

# Guardrails for NEW V1 BUY entries only
MIN_CASH_RESERVE = 750.0
MAX_OPEN_POSITIONS = 18
MAX_DEPLOYED_CAPITAL = 1800.0
MAX_ENTRY_BTC_EXPOSURE = 1800.0
MAX_DRAWDOWN_STOP = 0.15

LIVE_EXECUTION_ENABLED = False
EXECUTION_MODE = "BACKTEST / SHADOW READINESS"


## 1.1 Grid construction and V0-B initialization

In [ ]:
def validate_config():
    if INITIAL_CAPITAL <= 0 or GRID_GAP <= 0:
        raise ValueError("INITIAL_CAPITAL and GRID_GAP must be > 0.")
    if not (0 <= BUY_FEE < 1 and 0 <= SELL_FEE < 1):
        raise ValueError("Fees must be in [0, 1).")
    for name, floor, ceiling in [
        ("V0 baseline", V0_BASELINE_FLOOR, V0_BASELINE_CEILING),
        ("V1 active", V1_INITIAL_FLOOR, V1_INITIAL_CEILING),
    ]:
        if floor <= 0 or ceiling <= floor:
            raise ValueError(f"Invalid {name} range.")
        n = (ceiling - floor) / GRID_GAP
        if not np.isclose(n, round(n)):
            raise ValueError(f"{name} range must be divisible by GRID_GAP.")
    if RECENTER_TRIGGER_GRIDS <= 0:
        raise ValueError("RECENTER_TRIGGER_GRIDS must be > 0.")
    if not (0 <= MIN_CASH_RESERVE < INITIAL_CAPITAL):
        raise ValueError("Invalid MIN_CASH_RESERVE.")
    if MAX_OPEN_POSITIONS is not None and MAX_OPEN_POSITIONS <= 0:
        raise ValueError("Invalid MAX_OPEN_POSITIONS.")
    if MAX_DEPLOYED_CAPITAL is not None and MAX_DEPLOYED_CAPITAL <= 0:
        raise ValueError("Invalid MAX_DEPLOYED_CAPITAL.")
    if MAX_ENTRY_BTC_EXPOSURE is not None and MAX_ENTRY_BTC_EXPOSURE <= 0:
        raise ValueError("Invalid MAX_ENTRY_BTC_EXPOSURE.")
    if MAX_DRAWDOWN_STOP is not None and not (0 < MAX_DRAWDOWN_STOP < 1):
        raise ValueError("Invalid MAX_DRAWDOWN_STOP.")


validate_config()
V0_BASELINE_NUMBER_OF_GRIDS = int(round((V0_BASELINE_CEILING - V0_BASELINE_FLOOR) / GRID_GAP))
V1_NUMBER_OF_GRIDS = int(round((V1_INITIAL_CEILING - V1_INITIAL_FLOOR) / GRID_GAP))
V1_INITIAL_REFERENCE = (V1_INITIAL_FLOOR + V1_INITIAL_CEILING) / 2.0


def build_grid_template(floor, ceiling, gap):
    buy = np.arange(floor, ceiling, gap, dtype=float)
    return pd.DataFrame({
        "grid_slot": np.arange(1, len(buy) + 1),
        "grid_buy_price": buy,
        "sell_target": buy + gap,
    })


def derive_v0_initialization(grid, start_price, initial_capital, buy_fee):
    grid = grid.copy()
    if not (grid["grid_buy_price"].min() < start_price < grid["sell_target"].max()):
        raise ValueError("Start price must be inside the V0 baseline range.")

    seed = grid["sell_target"] > start_price
    reserve = ~seed
    seed_buy_prices = grid.loc[seed, "grid_buy_price"].to_numpy(float)

    funding_weight = int(reserve.sum()) + float(np.sum(start_price / seed_buy_prices))
    order_size = initial_capital / funding_weight

    grid["seed_at_start"] = seed
    grid["normal_order_size_usdt"] = order_size
    grid["normal_net_btc"] = order_size / grid["grid_buy_price"] * (1.0 - buy_fee)
    grid["initial_entry_cost_usdt"] = np.where(
        seed,
        grid["normal_net_btc"] / (1.0 - buy_fee) * start_price,
        0.0,
    )

    reserved_cash = float(reserve.sum() * order_size)
    seeded_cost = float(grid["initial_entry_cost_usdt"].sum())
    if not np.isclose(reserved_cash + seeded_cost, initial_capital, atol=1e-8):
        raise AssertionError("V0-B initialization does not reconcile.")

    return grid, {
        "normal_order_size_usdt": float(order_size),
        "funding_weight": float(funding_weight),
        "initial_sell_positions": int(seed.sum()),
        "initial_buy_levels": int(reserve.sum()),
        "reserved_cash_usdt": reserved_cash,
        "seeded_btc_cost_usdt": seeded_cost,
    }


def round_to_gap(price, gap):
    return float(np.floor(float(price) / gap + 0.5) * gap)


def build_dynamic_regime(reference, gap, number_of_grids, regime_id):
    lower = number_of_grids // 2
    upper = number_of_grids - lower
    floor = reference - lower * gap
    ceiling = reference + upper * gap
    if floor <= 0:
        raise ValueError("Dynamic floor must stay above zero.")
    buy = np.arange(floor, ceiling, gap, dtype=float)
    if len(buy) != number_of_grids:
        raise AssertionError("Dynamic regime grid count mismatch.")
    return {
        "regime_id": int(regime_id),
        "reference_price": float(reference),
        "floor": float(floor),
        "ceiling": float(ceiling),
        "buy_prices": buy,
        "sell_targets": buy + gap,
    }


V0_BASELINE_GRID = build_grid_template(V0_BASELINE_FLOOR, V0_BASELINE_CEILING, GRID_GAP)
V1_INITIAL_REGIME = build_dynamic_regime(V1_INITIAL_REFERENCE, GRID_GAP, V1_NUMBER_OF_GRIDS, 0)

print(f"V0 initialization range : {V0_BASELINE_FLOOR:,.0f} - {V0_BASELINE_CEILING:,.0f}")
print(f"V0 initialization grids : {V0_BASELINE_NUMBER_OF_GRIDS}")
print(f"V1 reference            : {V1_INITIAL_REFERENCE:,.0f}")
print(f"V1 active range         : {V1_INITIAL_REGIME['floor']:,.0f} - {V1_INITIAL_REGIME['ceiling']:,.0f}")
print(f"V1 active grids         : {V1_NUMBER_OF_GRIDS}")
print(f"Recenter trigger        : ±{RECENTER_TRIGGER_GRIDS * GRID_GAP:,.0f}")


## 1.2 Position and dynamic-grid engine

In [ ]:
def create_position(trade_id, regime_id, reference_price, buy_time, market_buy_price,
                    grid_buy_price, sell_target, normal_order_size, portfolio_value_at_buy,
                    buy_fee, sell_fee, entry_type):
    if entry_type == "INITIAL_SEED":
        target_net_btc = normal_order_size / grid_buy_price * (1.0 - buy_fee)
        gross_btc = target_net_btc / (1.0 - buy_fee)
        order_size_usdt = gross_btc * market_buy_price
    else:
        order_size_usdt = normal_order_size
        gross_btc = order_size_usdt / market_buy_price

    buy_fee_btc = gross_btc * buy_fee
    btc_amount = gross_btc - buy_fee_btc
    gross_sell_usdt = btc_amount * sell_target
    sell_fee_usdt = gross_sell_usdt * sell_fee

    return {
        "trade_id": int(trade_id), "regime_id": int(regime_id),
        "reference_price_at_buy": float(reference_price), "entry_type": entry_type,
        "status": "OPEN", "buy_time": buy_time, "buy_price": float(market_buy_price),
        "grid_buy_price": float(grid_buy_price), "sell_target": float(sell_target),
        "sell_time": pd.NaT, "order_size_usdt": float(order_size_usdt),
        "portfolio_value_at_buy": float(portfolio_value_at_buy),
        "order_pct_of_portfolio": float(order_size_usdt / portfolio_value_at_buy),
        "portfolio_value_at_sell": np.nan, "btc_amount": float(btc_amount),
        "buy_fee_btc": float(buy_fee_btc), "sell_fee_usdt": float(sell_fee_usdt),
        "net_sell_usdt": float(gross_sell_usdt - sell_fee_usdt), "net_pnl": np.nan,
    }


def initialize_from_v0(data, baseline_grid, initial_capital, buy_fee, sell_fee, v1_reference):
    start_time = data.iloc[0]["open_time"]
    start_price = float(data.iloc[0]["open"])
    initialized_grid, sizing = derive_v0_initialization(
        baseline_grid, start_price, initial_capital, buy_fee
    )

    positions, open_by_price, sell_heap, events = {}, {}, [], []
    cash, btc, deployed = float(initial_capital), 0.0, 0.0
    total_buy_fee_usdt, trade_id, event_id = 0.0, 0, 0

    for row in initialized_grid.loc[initialized_grid["seed_at_start"]].itertuples(index=False):
        portfolio_value = cash + btc * start_price
        trade_id += 1
        p = create_position(
            trade_id, -1, v1_reference, start_time, start_price,
            float(row.grid_buy_price), float(row.sell_target),
            float(sizing["normal_order_size_usdt"]), portfolio_value,
            buy_fee, sell_fee, "INITIAL_SEED"
        )
        cash_before, btc_before, deployed_before = cash, btc, deployed
        cash -= p["order_size_usdt"]
        btc += p["btc_amount"]
        deployed += p["order_size_usdt"]
        total_buy_fee_usdt += p["buy_fee_btc"] * start_price

        positions[trade_id] = p
        open_by_price[p["grid_buy_price"]] = trade_id
        heapq.heappush(sell_heap, (p["sell_target"], trade_id))

        event_id += 1
        events.append({
            "event_id": event_id, "time": start_time, "side": "BUY",
            "trade_id": trade_id, "regime_id": -1, "grid_buy_price": p["grid_buy_price"],
            "price": start_price, "cash_movement": -p["order_size_usdt"],
            "grid_cashflow": 0.0, "cash_before": cash_before, "cash_after": cash,
            "btc_before": btc_before, "btc_after": btc,
            "deployed_capital_before": deployed_before, "deployed_capital_after": deployed,
            "open_positions_after": len(open_by_price),
            "entry_exposure_after": btc * start_price, "initialization_trade": True,
        })

    initialization = {
        **sizing,
        "source": "FROZEN_V0_B_BASELINE",
        "baseline_floor": float(baseline_grid["grid_buy_price"].min()),
        "baseline_ceiling": float(baseline_grid["sell_target"].max()),
        "baseline_number_of_grids": int(len(baseline_grid)),
        "start_time": start_time, "start_price": start_price,
        "initial_cash": float(cash), "initial_btc": float(btc),
        "initial_btc_cost_usdt": float(deployed),
        "initial_buy_fee_usdt": float(total_buy_fee_usdt),
        "initial_btc_allocation_pct": float(deployed / initial_capital * 100.0),
        "initial_btc_market_value_usdt": float(btc * start_price),
    }

    return {
        "cash": cash, "btc": btc, "deployed_capital": deployed,
        "positions": positions, "open_by_grid_price": open_by_price,
        "sell_heap": sell_heap, "events": events, "trade_id": trade_id,
        "event_id": event_id, "initialization": initialization,
        "total_buy_fee_usdt": total_buy_fee_usdt,
    }


def run_dynamic_recenter_grid(data, baseline_grid, initial_capital, initial_reference, gap,
                              number_of_grids, recenter_trigger_grids, buy_fee, sell_fee,
                              min_cash_reserve, max_open_positions, max_deployed_capital,
                              max_entry_btc_exposure, max_drawdown_stop):
    data = data.sort_values("open_time").reset_index(drop=True).copy()
    if data.empty:
        raise ValueError("Market data is empty.")

    regime_id = 0
    regime = build_dynamic_regime(initial_reference, gap, number_of_grids, regime_id)
    state = initialize_from_v0(
        data, baseline_grid, initial_capital, buy_fee, sell_fee, initial_reference
    )

    cash, btc, deployed = state["cash"], state["btc"], state["deployed_capital"]
    positions = state["positions"]
    open_by_price = state["open_by_grid_price"]
    sell_heap, events = state["sell_heap"], state["events"]
    trade_id, event_id = state["trade_id"], state["event_id"]
    initialization = state["initialization"]

    # Fixed V0-B order size; no compounding.
    order_size = float(initialization["normal_order_size_usdt"])
    total_buy_fee_usdt = float(state["total_buy_fee_usdt"])
    total_sell_fee_usdt = realized_profit = 0.0
    completed_cycles = 0

    regime_history = [{
        "regime_id": 0, "effective_time": data.iloc[0]["open_time"],
        "reference_price": regime["reference_price"], "floor": regime["floor"],
        "ceiling": regime["ceiling"], "reason": "INITIAL_V1_ACTIVE_REGIME",
    }]
    recenter_events, risk_halt_events = [], []
    blocked = {
        "risk_halt": 0, "cash_reserve": 0, "max_open_positions": 0,
        "max_deployed_capital": 0, "max_entry_btc_exposure": 0,
    }

    n = len(data)
    equity_values = np.empty(n)
    cash_values = np.empty(n)
    btc_values = np.empty(n)
    deployed_values = np.empty(n)
    open_values = np.empty(n, dtype=int)
    btc_value_values = np.empty(n)
    reference_values = np.empty(n)
    regime_values = np.empty(n, dtype=int)
    risk_halt_values = np.empty(n, dtype=bool)

    previous_close, risk_halt, peak_equity = None, False, float(initial_capital)
    tolerance = 1e-12

    for i, candle in enumerate(data.itertuples(index=False)):
        timestamp = candle.open_time
        open_price, high_price, low_price, close_price = map(
            float, (candle.open, candle.high, candle.low, candle.close)
        )
        cash_at_candle_start = cash
        buy_budget = cash_at_candle_start
        sold_this_candle = set()

        # 1) SELL existing positions first.
        while sell_heap and sell_heap[0][0] <= high_price + tolerance:
            _, tid = heapq.heappop(sell_heap)
            p = positions.get(tid)
            if p is None or p["status"] != "OPEN":
                continue

            grid_price = p["grid_buy_price"]
            cash_before, btc_before, deployed_before = cash, btc, deployed
            portfolio_value = cash_before + btc_before * p["sell_target"]

            cash += p["net_sell_usdt"]
            btc -= p["btc_amount"]
            deployed -= p["order_size_usdt"]
            if abs(btc) < 1e-12: btc = 0.0
            if abs(deployed) < 1e-10: deployed = 0.0

            pnl = p["net_sell_usdt"] - p["order_size_usdt"]
            p.update(
                status="CLOSED", sell_time=timestamp,
                portfolio_value_at_sell=portfolio_value, net_pnl=float(pnl)
            )
            open_by_price.pop(grid_price, None)
            sold_this_candle.add(grid_price)
            total_sell_fee_usdt += p["sell_fee_usdt"]
            realized_profit += pnl
            completed_cycles += 1

            event_id += 1
            events.append({
                "event_id": event_id, "time": timestamp, "side": "SELL",
                "trade_id": tid, "regime_id": p["regime_id"], "grid_buy_price": grid_price,
                "price": p["sell_target"], "cash_movement": p["net_sell_usdt"],
                "grid_cashflow": pnl, "cash_before": cash_before, "cash_after": cash,
                "btc_before": btc_before, "btc_after": btc,
                "deployed_capital_before": deployed_before, "deployed_capital_after": deployed,
                "open_positions_after": len(open_by_price), "entry_exposure_after": np.nan,
                "initialization_trade": False,
            })

        # 2) BUY only on downward crossings of the active regime.
        downward_start = open_price if previous_close is None else max(previous_close, open_price)
        active = regime["buy_prices"]
        active_list = active.tolist()

        if low_price < downward_start:
            first_index = bisect.bisect_left(active_list, low_price)
            stop_index = bisect.bisect_left(active_list, downward_start)

            for k in range(stop_index - 1, first_index - 1, -1):
                grid_price = float(active[k])
                if grid_price in open_by_price or grid_price in sold_this_candle:
                    continue
                if risk_halt:
                    blocked["risk_halt"] += 1
                    break
                if buy_budget + tolerance < order_size:
                    break
                if buy_budget - order_size < min_cash_reserve - tolerance:
                    blocked["cash_reserve"] += 1
                    break
                if max_open_positions is not None and len(open_by_price) >= max_open_positions:
                    blocked["max_open_positions"] += 1
                    break
                if max_deployed_capital is not None and deployed + order_size > max_deployed_capital + tolerance:
                    blocked["max_deployed_capital"] += 1
                    break

                net_btc = order_size / grid_price * (1.0 - buy_fee)
                projected_btc = btc + net_btc
                projected_exposure = projected_btc * grid_price
                if max_entry_btc_exposure is not None and projected_exposure > max_entry_btc_exposure + tolerance:
                    blocked["max_entry_btc_exposure"] += 1
                    break

                portfolio_value = cash + btc * grid_price
                trade_id += 1
                p = create_position(
                    trade_id, regime["regime_id"], regime["reference_price"], timestamp,
                    grid_price, grid_price, grid_price + gap, order_size, portfolio_value,
                    buy_fee, sell_fee, "GRID_BUY"
                )
                cash_before, btc_before, deployed_before = cash, btc, deployed
                buy_budget -= order_size
                cash -= order_size
                btc += p["btc_amount"]
                deployed += order_size
                total_buy_fee_usdt += p["buy_fee_btc"] * grid_price

                positions[trade_id] = p
                open_by_price[grid_price] = trade_id
                heapq.heappush(sell_heap, (p["sell_target"], trade_id))

                event_id += 1
                events.append({
                    "event_id": event_id, "time": timestamp, "side": "BUY",
                    "trade_id": trade_id, "regime_id": regime["regime_id"],
                    "grid_buy_price": grid_price, "price": grid_price,
                    "cash_movement": -order_size, "grid_cashflow": 0.0,
                    "cash_before": cash_before, "cash_after": cash,
                    "btc_before": btc_before, "btc_after": btc,
                    "deployed_capital_before": deployed_before,
                    "deployed_capital_after": deployed,
                    "open_positions_after": len(open_by_price),
                    "entry_exposure_after": projected_exposure,
                    "initialization_trade": False,
                })

        # 3) End-of-candle risk state.
        equity = cash + btc * close_price
        peak_equity = max(peak_equity, equity)
        current_drawdown = equity / peak_equity - 1.0

        if (
            not risk_halt
            and max_drawdown_stop is not None
            and current_drawdown <= -max_drawdown_stop
        ):
            risk_halt = True
            risk_halt_events.append({
                "decision_time": timestamp,
                "effective_time": data.iloc[i + 1]["open_time"] if i + 1 < n else pd.NaT,
                "drawdown": float(current_drawdown), "equity": float(equity),
                "peak_equity": float(peak_equity),
            })

        equity_values[i] = equity
        cash_values[i] = cash
        btc_values[i] = btc
        deployed_values[i] = deployed
        open_values[i] = len(open_by_price)
        btc_value_values[i] = btc * close_price
        reference_values[i] = regime["reference_price"]
        regime_values[i] = regime["regime_id"]
        risk_halt_values[i] = risk_halt

        # 4) Re-center at close; effective next candle.
        trigger = recenter_trigger_grids * gap
        if close_price >= regime["reference_price"] + trigger or close_price <= regime["reference_price"] - trigger:
            new_reference = round_to_gap(close_price, gap)
            if new_reference != regime["reference_price"]:
                old = regime
                regime_id += 1
                regime = build_dynamic_regime(new_reference, gap, number_of_grids, regime_id)
                direction = "UP" if new_reference > old["reference_price"] else "DOWN"
                event = {
                    "decision_time": timestamp,
                    "effective_time": data.iloc[i + 1]["open_time"] if i + 1 < n else pd.NaT,
                    "direction": direction, "close": close_price,
                    "old_reference": old["reference_price"], "new_reference": new_reference,
                    "old_floor": old["floor"], "old_ceiling": old["ceiling"],
                    "new_floor": regime["floor"], "new_ceiling": regime["ceiling"],
                }
                recenter_events.append(event)
                regime_history.append({
                    "regime_id": regime_id, "effective_time": event["effective_time"],
                    "reference_price": regime["reference_price"], "floor": regime["floor"],
                    "ceiling": regime["ceiling"], "reason": f"RECENTER_{direction}",
                })

        previous_close = close_price

    equity_curve = pd.DataFrame({
        "open_time": data["open_time"], "close": data["close"],
        "cash": cash_values, "btc": btc_values, "btc_market_value": btc_value_values,
        "deployed_capital": deployed_values, "open_positions": open_values,
        "equity": equity_values, "reference_price": reference_values,
        "regime_id": regime_values, "risk_halt": risk_halt_values,
    })

    return {
        "data": data, "equity_curve": equity_curve, "positions": positions,
        "events": pd.DataFrame(events), "initialization": initialization,
        "regime_history": regime_history, "recenter_events": recenter_events,
        "risk_halt_events": risk_halt_events, "blocked_entries": blocked,
        "completed_cycles": completed_cycles, "realized_profit": float(realized_profit),
        "total_buy_fee_usdt": float(total_buy_fee_usdt),
        "total_sell_fee_usdt": float(total_sell_fee_usdt),
        "normal_order_size_usdt": order_size, "final_cash": float(cash),
        "final_btc": float(btc), "final_deployed_capital": float(deployed),
        "risk_halt": bool(risk_halt),
    }


# 2. Backtest System

In [ ]:
DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"
TIMEFRAME = "1m"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"


def load_market_data(symbol, timeframe, data_dir, start_date, end_date):
    path = os.path.join(data_dir, f"{symbol}-{timeframe}-combined.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    required = {"open_time", "open", "high", "low", "close", "volume"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    df["open_time"] = pd.to_datetime(df["open_time"], utc=True)
    df[["open", "high", "low", "close", "volume"]] = df[
        ["open", "high", "low", "close", "volume"]
    ].astype(float)

    start_ts = pd.Timestamp(start_date, tz="UTC")
    end_ts = pd.Timestamp(end_date, tz="UTC")
    return (
        df.drop_duplicates("open_time").sort_values("open_time")
        .loc[lambda x: (x["open_time"] >= start_ts) & (x["open_time"] < end_ts)]
        .reset_index(drop=True)
    )


def performance_stats(data, equity, initial_capital):
    equity = np.asarray(equity, dtype=float)
    peak = np.maximum.accumulate(equity)
    drawdown = equity / peak - 1.0
    final_equity = float(equity[-1])
    net_return = final_equity / initial_capital - 1.0
    max_drawdown = float(drawdown.min())

    elapsed_days = (data["open_time"].iloc[-1] - data["open_time"].iloc[0]).total_seconds() / 86400.0
    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        growth = np.log(final_equity / initial_capital) * (365.25 / elapsed_days)
        if growth < 700:
            annualized_return = float(np.expm1(growth))

    calmar = np.nan
    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar = float(annualized_return / abs(max_drawdown))

    return {
        "final_equity": final_equity, "net_return": float(net_return),
        "annualized_return": annualized_return, "max_drawdown": max_drawdown,
        "calmar_ratio": calmar, "drawdown": drawdown,
    }


def build_buy_hold_benchmark(data, initial_capital, buy_fee):
    entry_price = float(data.iloc[0]["open"])
    final_price = float(data.iloc[-1]["close"])
    gross_btc = initial_capital / entry_price
    net_btc = gross_btc * (1.0 - buy_fee)
    equity = net_btc * data["close"].to_numpy(float)
    stats = performance_stats(data, equity, initial_capital)
    return {
        "entry_price": entry_price, "final_price": final_price,
        "gross_btc": float(gross_btc), "net_btc": float(net_btc),
        "entry_fee_usdt_equiv": float(gross_btc * buy_fee * entry_price),
        "final_equity": stats["final_equity"], "net_return": stats["net_return"],
        "annualized_return": stats["annualized_return"],
        "max_drawdown": stats["max_drawdown"], "calmar_ratio": stats["calmar_ratio"],
    }


def build_trade_history(positions, final_time, final_close):
    rows = []
    for trade_id in sorted(positions):
        p = positions[trade_id]
        if p["status"] == "CLOSED":
            net_pnl = float(p["net_pnl"])
            holding_time = p["sell_time"] - p["buy_time"]
        else:
            net_pnl = p["btc_amount"] * final_close - p["order_size_usdt"]
            holding_time = final_time - p["buy_time"]

        rows.append({
            "Trade ID": p["trade_id"], "Regime ID": p["regime_id"],
            "Entry Type": p["entry_type"], "Status": p["status"],
            "Buy Time": p["buy_time"], "Buy Price": p["buy_price"],
            "Grid Buy Price": p["grid_buy_price"],
            "Reference at Buy": p["reference_price_at_buy"],
            "Sell Target": p["sell_target"], "Sell Time": p["sell_time"],
            "Order Size (USDT)": p["order_size_usdt"],
            "Portfolio Value at Buy": p["portfolio_value_at_buy"],
            "Order % of Portfolio": p["order_pct_of_portfolio"],
            "Portfolio Value at Sell": p["portfolio_value_at_sell"],
            "Net P&L": net_pnl, "Holding Time": holding_time,
        })
    return pd.DataFrame(rows)


def audit_v1(result):
    positions = result["positions"]
    events = result["events"]
    curve = result["equity_curve"]
    init = result["initialization"]

    open_positions = [p for p in positions.values() if p["status"] == "OPEN"]
    closed_positions = [p for p in positions.values() if p["status"] == "CLOSED"]
    final_close = float(result["data"].iloc[-1]["close"])

    seed_quantity_ok = all(
        np.isclose(
            p["btc_amount"],
            result["normal_order_size_usdt"] / p["grid_buy_price"] * (1.0 - BUY_FEE),
            atol=1e-12,
        )
        for p in positions.values() if p["entry_type"] == "INITIAL_SEED"
    )
    fixed_order_ok = all(
        np.isclose(p["order_size_usdt"], result["normal_order_size_usdt"], atol=1e-9)
        for p in positions.values() if p["entry_type"] == "GRID_BUY"
    )

    same_candle_rebuy_ok = True
    if len(events):
        sells = events.loc[events["side"].eq("SELL"), ["time", "grid_buy_price"]]
        buys = events.loc[
            events["side"].eq("BUY") & ~events["initialization_trade"].fillna(False),
            ["time", "grid_buy_price"],
        ]
        if len(sells) and len(buys):
            same_candle_rebuy_ok = sells.merge(
                buys, on=["time", "grid_buy_price"], how="inner"
            ).empty

    new_buys = events.loc[
        events["side"].eq("BUY") & ~events["initialization_trade"].fillna(False)
    ].copy() if len(events) else pd.DataFrame()

    guardrail_cash_ok = True
    guardrail_open_ok = True
    guardrail_deployed_ok = True
    guardrail_exposure_ok = True
    if len(new_buys):
        guardrail_cash_ok = bool((new_buys["cash_after"] >= MIN_CASH_RESERVE - 1e-8).all())
        if MAX_OPEN_POSITIONS is not None:
            guardrail_open_ok = bool((new_buys["open_positions_after"] <= MAX_OPEN_POSITIONS).all())
        if MAX_DEPLOYED_CAPITAL is not None:
            guardrail_deployed_ok = bool(
                (new_buys["deployed_capital_after"] <= MAX_DEPLOYED_CAPITAL + 1e-8).all()
            )
        if MAX_ENTRY_BTC_EXPOSURE is not None:
            guardrail_exposure_ok = bool(
                (new_buys["entry_exposure_after"] <= MAX_ENTRY_BTC_EXPOSURE + 1e-8).all()
            )

    checks = {
        "initial_capital_reconciliation": bool(np.isclose(
            init["initial_cash"] + init["initial_btc_cost_usdt"], INITIAL_CAPITAL, atol=1e-8
        )),
        "v0_baseline_source_confirmed": init["source"] == "FROZEN_V0_B_BASELINE",
        "cash_never_negative": bool((curve["cash"] >= -1e-8).all()),
        "btc_never_negative": bool((curve["btc"] >= -1e-12).all()),
        "final_equity_identity": bool(np.isclose(
            curve["equity"].iloc[-1],
            result["final_cash"] + result["final_btc"] * final_close,
            atol=1e-8,
        )),
        "final_btc_matches_open_positions": bool(np.isclose(
            result["final_btc"], sum(p["btc_amount"] for p in open_positions), atol=1e-12
        )),
        "deployed_capital_matches_open_cost": bool(np.isclose(
            result["final_deployed_capital"],
            sum(p["order_size_usdt"] for p in open_positions),
            atol=1e-8,
        )),
        "closed_trade_count_reconciliation": result["completed_cycles"] == len(closed_positions),
        "initial_seed_quantity_matches_v0_grid": seed_quantity_ok,
        "fixed_order_size_matches_v0_no_compounding": fixed_order_ok,
        "no_same_candle_sell_rebuy": same_candle_rebuy_ok,
        "new_buy_cash_reserve_respected": guardrail_cash_ok,
        "new_buy_max_open_positions_respected": guardrail_open_ok,
        "new_buy_max_deployed_capital_respected": guardrail_deployed_ok,
        "new_buy_max_entry_btc_exposure_respected": guardrail_exposure_ok,
    }

    return {
        "status": "PASS" if all(checks.values()) else "FAIL",
        "checks": checks,
        "diagnostics": {
            "guardrail_scope": "NEW BUY entries only",
            "initial_open_positions": init["initial_sell_positions"],
            "initial_deployed_capital": init["initial_btc_cost_usdt"],
            "initial_btc_market_value": init["initial_btc_market_value_usdt"],
            "max_open_positions_observed_all_positions": int(curve["open_positions"].max()),
            "max_deployed_capital_observed_all_positions": float(curve["deployed_capital"].max()),
            "max_btc_market_value_observed_all_positions": float(curve["btc_market_value"].max()),
            "new_buy_event_count": int(len(new_buys)),
        },
    }


## 2.1 Run and review

In [ ]:
data = load_market_data(SYMBOL, TIMEFRAME, DATA_DIR, START_DATE, END_DATE)

result = run_dynamic_recenter_grid(
    data=data, baseline_grid=V0_BASELINE_GRID, initial_capital=INITIAL_CAPITAL,
    initial_reference=V1_INITIAL_REFERENCE, gap=GRID_GAP,
    number_of_grids=V1_NUMBER_OF_GRIDS,
    recenter_trigger_grids=RECENTER_TRIGGER_GRIDS,
    buy_fee=BUY_FEE, sell_fee=SELL_FEE,
    min_cash_reserve=MIN_CASH_RESERVE,
    max_open_positions=MAX_OPEN_POSITIONS,
    max_deployed_capital=MAX_DEPLOYED_CAPITAL,
    max_entry_btc_exposure=MAX_ENTRY_BTC_EXPOSURE,
    max_drawdown_stop=MAX_DRAWDOWN_STOP,
)

v1_stats = performance_stats(
    result["data"], result["equity_curve"]["equity"].to_numpy(float), INITIAL_CAPITAL
)
result["equity_curve"]["drawdown"] = v1_stats["drawdown"]
buy_hold = build_buy_hold_benchmark(result["data"], INITIAL_CAPITAL, BUY_FEE)

final_time = result["data"].iloc[-1]["open_time"]
final_close = float(result["data"].iloc[-1]["close"])
trade_history = build_trade_history(result["positions"], final_time, final_close)
open_positions = [p for p in result["positions"].values() if p["status"] == "OPEN"]

summary = {
    "initial_capital": INITIAL_CAPITAL,
    "final_equity": v1_stats["final_equity"],
    "net_return": v1_stats["net_return"],
    "annualized_return": v1_stats["annualized_return"],
    "max_drawdown": v1_stats["max_drawdown"],
    "calmar_ratio": v1_stats["calmar_ratio"],
    "completed_cycles": result["completed_cycles"],
    "open_positions": len(open_positions),
    "final_cash": result["final_cash"],
    "final_btc": result["final_btc"],
    "final_deployed_capital": result["final_deployed_capital"],
    "realized_profit": result["realized_profit"],
    "unrealized_pnl": float(sum(
        p["btc_amount"] * final_close - p["order_size_usdt"] for p in open_positions
    )),
    "total_fee_usdt_equiv": result["total_buy_fee_usdt"] + result["total_sell_fee_usdt"],
    "recenter_count": len(result["recenter_events"]),
    "risk_halt": result["risk_halt"],
    "blocked_entries": result["blocked_entries"],
}

comparison_vs_buy_hold = {
    "final_equity_difference_usdt": summary["final_equity"] - buy_hold["final_equity"],
    "excess_return": summary["net_return"] - buy_hold["net_return"],
    "drawdown_improvement": abs(buy_hold["max_drawdown"]) - abs(summary["max_drawdown"]),
    "calmar_difference": summary["calmar_ratio"] - buy_hold["calmar_ratio"],
}

audit = audit_v1(result)

print("=== V1 Dynamic Re-centering Grid ===")
for k, v in summary.items(): print(f"{k}: {v}")

print("\n=== Initialization (must match V0-B) ===")
for k, v in result["initialization"].items(): print(f"{k}: {v}")

print("\n=== BTC Buy & Hold ===")
for k, v in buy_hold.items(): print(f"{k}: {v}")

print("\n=== V1 vs Buy & Hold ===")
for k, v in comparison_vs_buy_hold.items(): print(f"{k}: {v}")

print("\n=== Audit ===")
print(audit["status"])
for k, v in audit["checks"].items(): print(f"{k}: {v}")

print("\n=== Audit Diagnostics ===")
for k, v in audit["diagnostics"].items(): print(f"{k}: {v}")

print(f"\nRe-center count: {len(result['recenter_events'])}")
if result["recenter_events"]:
    display(pd.DataFrame(result["recenter_events"]).head(20))


## 2.2 Trade History

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
display(trade_history.head(100))
print(f"Total trades: {len(trade_history)}")
print(f"Closed      : {(trade_history['Status'] == 'CLOSED').sum()}")
print(f"Open        : {(trade_history['Status'] == 'OPEN').sum()}")


# 3. Immutable Logging

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return [json_safe(v) for v in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        value = float(value)
        return value if np.isfinite(value) else None
    if isinstance(value, (pd.Timestamp, datetime)):
        return None if pd.isna(value) else value.isoformat()
    if isinstance(value, pd.Timedelta):
        return str(value)
    if value is pd.NaT:
        return None
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    if isinstance(value, (bool, str, int)) or value is None:
        return value
    return str(value)


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = f"logs/v1/{RUN_ID}"
SUMMARY_PATH = f"{RUN_DIR}/summary.json"
TRADE_HISTORY_PATH = f"{RUN_DIR}/trade_history.csv"

log_payload = json_safe({
    "log_schema_version": 4,
    "run_info": {
        "run_id": RUN_ID,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "strategy": "V1 Dynamic Re-centering Grid",
        "comparison_design": "Same frozen V0-B initial portfolio and fixed order size",
        "execution_mode": EXECUTION_MODE,
        "live_execution_enabled": LIVE_EXECUTION_ENABLED,
    },
    "v0_baseline_initialization_config": {
        "floor": V0_BASELINE_FLOOR, "ceiling": V0_BASELINE_CEILING,
        "gap": GRID_GAP, "number_of_grids": V0_BASELINE_NUMBER_OF_GRIDS,
        "initial_capital": INITIAL_CAPITAL,
    },
    "v1_dynamic_grid_config": {
        "initial_floor": V1_INITIAL_FLOOR, "initial_ceiling": V1_INITIAL_CEILING,
        "initial_reference": V1_INITIAL_REFERENCE, "grid_gap": GRID_GAP,
        "number_of_grids": V1_NUMBER_OF_GRIDS,
        "recenter_trigger_grids": RECENTER_TRIGGER_GRIDS,
        "recenter_trigger_usdt": RECENTER_TRIGGER_GRIDS * GRID_GAP,
        "buy_fee": BUY_FEE, "sell_fee": SELL_FEE,
        "normal_order_size_usdt": result["normal_order_size_usdt"],
        "normal_order_size_source": "V0-B baseline",
        "no_compounding": True, "old_positions_survive_recenter": True,
    },
    "risk_guardrails": {
        "scope": "NEW BUY entries only; inherited V0-B initial portfolio may start above these limits",
        "min_cash_reserve": MIN_CASH_RESERVE,
        "max_open_positions": MAX_OPEN_POSITIONS,
        "max_deployed_capital": MAX_DEPLOYED_CAPITAL,
        "max_entry_btc_exposure": MAX_ENTRY_BTC_EXPOSURE,
        "max_drawdown_stop": MAX_DRAWDOWN_STOP,
        "drawdown_halt_behavior": "one-way halt of NEW BUY entries; existing positions may still SELL",
    },
    "backtest_config": {
        "symbol": SYMBOL, "timeframe": TIMEFRAME,
        "start_date": START_DATE, "end_date": END_DATE,
        "data_rows": len(result["data"]),
        "first_candle": result["data"].iloc[0]["open_time"],
        "last_candle": result["data"].iloc[-1]["open_time"],
    },
    "initialization": result["initialization"],
    "summary": summary,
    "buy_hold_benchmark": buy_hold,
    "comparison_vs_buy_hold": comparison_vs_buy_hold,
    "recenter_events": result["recenter_events"],
    "risk_halt_events": result["risk_halt_events"],
    "audit": audit,
    "trade_history_file": TRADE_HISTORY_PATH,
})

print(f"Run ID: {RUN_ID}")
print(f"Summary: {SUMMARY_PATH}")
print(f"Trade history: {TRADE_HISTORY_PATH}")


## 3.1 Upload logs

In [ ]:
from google.colab import userdata


def create_github_file(repo, branch, path, text_content, token, message, max_retries=3):
    api_url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    existing = requests.get(api_url, headers=headers, params={"ref": branch}, timeout=30)
    if existing.status_code == 200:
        raise FileExistsError(f"Immutable log path already exists: {path}")
    if existing.status_code != 404:
        existing.raise_for_status()

    body = {
        "message": message,
        "content": base64.b64encode(text_content.encode("utf-8")).decode("ascii"),
        "branch": branch,
    }

    for attempt in range(1, max_retries + 1):
        response = requests.put(api_url, headers=headers, json=body, timeout=30)
        if response.status_code in (200, 201):
            payload = response.json()
            return {
                "commit_sha": payload["commit"]["sha"],
                "content_sha": payload["content"]["sha"],
                "path": path,
            }
        if response.status_code == 409 and attempt < max_retries:
            time.sleep(attempt)
            continue
        response.raise_for_status()


github_token = userdata.get("GITHUB_TOKEN")

summary_upload = create_github_file(
    "natdanaiii/Trading", "main", SUMMARY_PATH,
    json.dumps(log_payload, indent=2, allow_nan=False),
    github_token, f"Add V1 backtest summary {RUN_ID}",
)
trade_history_upload = create_github_file(
    "natdanaiii/Trading", "main", TRADE_HISTORY_PATH,
    trade_history.to_csv(index=False),
    github_token, f"Add V1 trade history {RUN_ID}",
)

print("GitHub immutable log upload: SUCCESS")
print(summary_upload)
print(trade_history_upload)
